### Download Survival Data


In [5]:
import requests, json

TEST_BIOSAMPLE_ID = "pgxbs-kftvgl5h"   
BASE = f"https://progenetix.org/beacon/biosamples/{TEST_BIOSAMPLE_ID}/g_variants/"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def run_query(params, label="query"):
    r = SESSION.get(BASE, params=params, timeout=120)
    print(f"--- {label} ---")
    print("status:", r.status_code)
    print("url:", r.url)
    print("content-type:", r.headers.get("Content-Type"))
    try:
        data = r.json()
        print(json.dumps(data, indent=2)[:2000])
        return r, data
    except Exception:
        print(r.text[:2000])
        return r, None

In [6]:
params = {
    "geneId": "EGFR",
    "requestedGranularity": "boolean"
}

r_bool, data_bool = run_query(params, label="biosample g_variants + geneId + boolean")

--- biosample g_variants + geneId + boolean ---
status: 200
url: https://progenetix.org/beacon/biosamples/pgxbs-kftvgl5h/g_variants/?geneId=EGFR&requestedGranularity=boolean
content-type: application/json
{
  "meta": {
    "apiVersion": "v2.3.0-beaconplus",
    "beaconId": "org.progenetix",
    "receivedRequestSummary": {
      "apiVersion": "v2.3.0-beaconplus",
      "beaconId": "org.progenetix",
      "datasetIds": [
        "progenetix"
      ],
      "includeResultsetResponses": "HIT",
      "pagination": {
        "limit": 200,
        "skip": 0
      },
      "requestedGranularity": "boolean",
      "requestedSchemas": [
        {
          "id": "genomicVariant",
          "name": "Default schema for a genomic variation",
          "referenceToSchemaDefinition": "https://progenetix.org/services/schemas/genomicVariant",
          "schemaVersion": "v2.2.0"
        }
      ],
      "testMode": false
    },
    "returnedGranularity": "boolean",
    "returnedSchemas": [
      {
     

In [19]:
import requests
import pandas as pd
from io import StringIO

# On définit les URLs pour les types de cancer (exemples de Progenetix)
sampletable_urls = {
    "TCGA_ESCA": "https://progenetix.org/services/sampletable/?filters=pgx:TCGA-ESCA",

}

In [3]:
# Download sample tables and store them in a dictionary.

sampletables = {}

for source_query, url in sampletable_urls.items():
    print(f"Downloading sample table for {source_query}")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    df_tmp = pd.read_csv(StringIO(response.text), sep="\t")
    df_tmp["source_query"] = source_query
    sampletables[source_query] = df_tmp

In [4]:
#if many tables, concatenate them into a single DataFrame
all_samples = pd.concat(sampletables.values(), ignore_index=True)

In [5]:
#name of the columns
print(all_samples.columns)

# Clean table with just the column needed for survival analysis
survival_Ondata = all_samples[['biosample_id', 'followup_days', 'biosample_status_id']].copy()

#  Drop the data withotut followup days, as they cannot be used for survival analysis
survival_data = survival_Ondata.dropna(subset=['followup_days'])

Index(['biosample_id', 'individual_id', 'biosample_name', 'notes',
       'histological_diagnosis_id', 'histological_diagnosis_label',
       'pathological_stage_id', 'pathological_stage_label',
       'biosample_status_id', 'biosample_status_label',
       'sample_origin_type_id', 'sample_origin_type_label',
       'sampled_tissue_id', 'sampled_tissue_label', 'tnm', 'tumor_grade_id',
       'tumor_grade_label', 'age_iso', 'age_days', 'icdo_morphology_id',
       'icdo_morphology_label', 'icdo_topography_id', 'icdo_topography_label',
       'pubmed_id', 'pubmed_label', 'cellosaurus_id', 'cellosaurus_label',
       'cbioportal_id', 'cbioportal_label', 'tcgaproject_id',
       'tcgaproject_label', 'cohorts', 'geoprov_id', 'geoprov_city',
       'geoprov_country', 'geoprov_iso_alpha3', 'geoprov_long_lat',
       'followup_days', 'sex_id', 'sex_label', 'group_id', 'group_label',
       'source_query'],
      dtype='str')


In [ ]:
""" I need to find the status alive or dead """
"""# i"m using biosample_status_id normally it has 'EFO:0009654', 'EFO:0009656' but not sure if it's survival or death
# Dictionnary for the mapping
#  0 = Vivant (Censured), 1 = Décédé (Event)
status_map = {
    'EFO:0009654': 0, # Alive
    'EFO:0009656': 1  # Dead
}

# transformation
survival_data['status'] = survival_data['biosample_status_id'].map(status_map)

# Verification of the transformation, looks if there are any remaining empty values
survival_data = survival_data.dropna(subset=['status'])
print(survival_data.head())"""

_IncompleteInputError: incomplete input (144174633.py, line 1)

In [ ]:
# Configuration of API Progenetix / Beacon
BIOSAMPLES_BASE_URL = "https://progenetix.org/beacon/biosamples/"
TIMEOUT = 120

# Création of a session (to be more efficient for making many requests)
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0"})

In [11]:
def fetch_variants_for_biosample(biosample_id):
    # Download all variant records for one biosample.
    url = f"{BIOSAMPLES_BASE_URL}{biosample_id}/g_variants/"
    r = SESSION.get(url, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json(), r.url

In [ ]:
# list to stock evry variants
all_variants_records = []

# list of the biosample IDs (testing on the first 20 for now)
biosample_ids = all_samples['biosample_id'].unique()[:20] 

print(f"Extraction des variants pour {len(biosample_ids)} échantillons...")

for bs_id in biosample_ids:
    try:
        # call function 
        data, url = fetch_variants_for_biosample(bs_id)
        
        # Extraction of the results list 
        results = data.get("response", {}).get("resultSets", [{}])[0].get("results", [])
        
        # Add each variant found to our collection
        for variant in results:
            variant_entry = {
                "biosample_id": bs_id,
                "variant_id": variant.get("variantInternalId"),
                "type": variant.get("variantType"), # Utile pour le 'gain' de ton schéma
                "raw_info": str(variant) # Pour chercher le gène plus tard
            }
            all_variants_records.append(variant_entry)
            
    except Exception as e:
        print(f"Erreur sur l'ID {bs_id} : {e}")

# transformation in dataframe
df_variants = pd.DataFrame(all_variants_records)

print(f"Phase terminée ! Nous avons récupéré {len(df_variants)} variants au total.")
print(df_variants.head())

Extraction des variants pour 20 échantillons...
Phase terminée ! Nous avons récupéré 1867 variants au total.
     biosample_id                                    variant_id  type  \
0  pgxbs-kftvhpgy    NC_000004.12:69873279-69873993:EFO_0030068  None   
1  pgxbs-kftvhpgy    NC_000010.11:12660069-12661941:EFO_0030068  None   
2  pgxbs-kftvhpgy    NC_000010.11:71365606-71366970:EFO_0030068  None   
3  pgxbs-kftvhpgy  NC_000004.12:162445156-162445364:EFO_0030068  None   
4  pgxbs-kftvhpgy    NC_000019.10:11057986-11058376:EFO_0030068  None   

                                            raw_info  
0  {'caseLevelData': [{'analysisId': 'pgxcs-kftvz...  
1  {'caseLevelData': [{'analysisId': 'pgxcs-kftvz...  
2  {'caseLevelData': [{'analysisId': 'pgxcs-kftvz...  
3  {'caseLevelData': [{'analysisId': 'pgxcs-kftvz...  
4  {'caseLevelData': [{'analysisId': 'pgxcs-kftvz...  


In [ ]:
# download the gene panel
gene_panel_df = pd.read_csv('/Users/bgadmin/Downloads/gene_cnv_cancer_panel', sep='\t')

# On extrait la colonne qui contient les symboles des gènes (ex: 'symbol' ou 'gene_symbol')
# Vérifie le nom de la colonne dans ton fichier TSV et remplace 'gene_symbol' si besoin
liste_genes = gene_panel_df['gene_symbol'].unique().tolist()

print(f"Panel chargé : {len(liste_genes)} gènes à analyser.")

In [ ]:
matrice_resultats = []

# go through the biosamples already identified.
for bs_id in survival_data['biosample_id'].unique():
    # Initialisation du patient : tout à 0 (no-hit)
    patient_data = {"biosample_id": bs_id}
    for gene in liste_genes:
        patient_data[f"{gene}_gain"] = 0 
        
    try:
        # Call the API for this biosample
        variants_list, _ = fetch_variants_for_biosample(bs_id)
        
        # Scan the received variants
        for v in variants_list:
            v_str = str(v)
            v_type = v.get("variantType", "")
            
            #  only check the genes in your panel
            for gene in liste_genes:
                # If the gene is in the variant AND it's a gain (DUP or AMP)
                if gene in v_str and v_type in ["DUP", "AMP"]:
                    patient_data[f"{gene}_gain"] = 1
                    
    except Exception as e:
        print(f"Erreur pour {bs_id}: {e}")
        
    matrice_resultats.append(patient_data)

# Creation of the final DataFrame of hits
df_gene_hits = pd.DataFrame(matrice_resultats)